# Training & Evaluating the model

## 0. Imports & Config

In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
import joblib

ARTIFACT_PATH = r"C:\Users\User\Documents\Key Future Docs\Projects\Phising URL\artifacts\artifact.joblib"
RANDOM_STATE = 42

COLS_TO_DROP = ['url', 'domain', 'subdomains', 'path', 'tld', 'protocol']

## 1. Load & Prepare the data for training

### A. Load artifact

In [2]:
artifact = joblib.load(ARTIFACT_PATH)
train_df = artifact["train_df"]
test_df = artifact["test_df"]
train_ngrams = artifact["train_ngrams"]
test_ngrams = artifact["test_ngrams"]
vectorizer = artifact["vectorizer"]
vectorizer_cols = artifact["vectorizer_cols"]

print(f"Train df shape: {train_df.shape}, Test df shape: {test_df.shape}")
print(f"Train ngrams shape: {train_ngrams.shape}, Test ngrams shape: {test_ngrams.shape}")
train_df.head()

Train df shape: (657593, 21), Test df shape: (164399, 21)
Train ngrams shape: (657593, 6000), Test ngrams shape: (164399, 6000)


,url,is_phishing,protocol,domain,subdomains,path,tld,is_ip,url_length,domain_length,...,path_length,tld_length,dot_count,domain_dot_count,digit_count,word_count,special_char_count,digit_letter_ratio,hyphen_count,contains_words_count
329413,answers.yahoo.com/question/index?qid=201008261...,True,http,answers.yahoo.com,"[answers, yahoo]",/question/index,com,False,58,17,...,15,3,2,2,15,7,6,2.142857,0,0
518253,facebook.com/people/Kathryn-Pizzino/1182562450,True,http,facebook.com,[facebook],/people/Kathryn-Pizzino/1182562450,com,False,46,12,...,34,3,1,1,10,6,5,1.666667,0,0
794241,ngv-shop.de,False,http,ngv-shop.de,[ngv-shop],,de,False,11,11,...,0,2,1,1,0,3,2,0.000000,1,0
503708,duplessis-travelguide.com/La-Minganie/ville_ba...,True,http,duplessis-travelguide.com,[duplessis-travelguide],/La-Minganie/ville_baie-johan-beetz_2880.php,com,False,69,25,...,44,3,2,1,4,9,10,0.444444,1,0
150185,https://rebrand.ly/fcbc36,False,https,rebrand.ly,[rebrand],/fcbc36,ly,False,25,10,...,7,2,1,1,2,4,5,0.500000,0,0


### B. Compress n-grams with Single Vector Decomposition (SVD)

In [3]:
svd = TruncatedSVD(n_components=100, random_state=RANDOM_STATE)
X_ngram_train = svd.fit_transform(train_ngrams)
X_ngram_test = svd.transform(test_ngrams)

### C. Build stack for training

In [4]:
X_train = train_df.drop(columns=COLS_TO_DROP + ['is_phishing']).to_numpy()
X_test = test_df.drop(columns=COLS_TO_DROP + ['is_phishing']).to_numpy()

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [5]:
X_train = np.hstack([X_train, X_ngram_train])
y_train = train_df['is_phishing']

X_test = np.hstack([X_test, X_ngram_test])
y_test = test_df['is_phishing']

## 2. Train an MLP

In [6]:
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    max_iter=50,
    random_state=RANDOM_STATE,
    early_stopping=True,
)

mlp.fit(X_train, y_train)
print(f"Trained in {mlp.n_iter_} iterations")

Trained in 32 iterations


In [7]:
preds = mlp.predict(X_test)
probs = mlp.predict_proba(X_test) # probs has 2 columns, first is probability of being 0, second is probability of being 1

accuracy = accuracy_score(y_test, preds)
auc = roc_auc_score(y_test, probs[:, 1])

## 3. Evalutate the model

In [8]:
print("Classification Report...")
print(classification_report(y_test, preds))

print("Confusion Matrix...")
print(confusion_matrix(y_test, preds))

Classification Report...
              precision    recall  f1-score   support

       False       0.97      0.97      0.97     78996
        True       0.97      0.97      0.97     85403

    accuracy                           0.97    164399
   macro avg       0.97      0.97      0.97    164399
weighted avg       0.97      0.97      0.97    164399

Confusion Matrix...
[[76621  2375]
 [ 2205 83198]]
